In [1]:
# %load_ext autoreload
# %autoreload 2

# autoreload re-imports src/vectorization/phase4/ modules before every cell run,
# so edits to the pipeline source take effect without restarting the kernel.
# Model checkpoints are cached in _R2G_MODEL_CACHE / _SEG_MODEL_CACHE —
# intentional: only the code should reload, not the weights.

# Phase 4 — Graph-to-Vector Vectorization

Set `INPUT_IMAGE_PATH` in the **last cell** to any floor plan image, then **Run All Cells**.

Requires `graph_pred.json` to already exist alongside the input image (produced by
`phase4_raster2graph.ipynb`). If it is absent the pipeline falls back to running
R2G inference automatically (requires GPU + R2G checkpoint).

All output files are written to the **same folder as the input image**:

| File | Description |
|---|---|
| `input.png` | preprocessed 512×512 canvas |
| `image_segmentation.png` | 7-class segmentation preview |
| `image_debug_overlay.png` | audit overlay (graph + openings + rejections) |
| `graph_pred.svg` | raw R2G wall graph SVG |
| `graph_pred.json` | raw R2G wall graph JSON |
| `graph_overlay.png` | raw graph on input |
| `graph_overlay_aligned.png` | orthogonally aligned graph on input |
| `final_vector.svg` | final wall/window/door SVG |
| `final_vector.json` | typed geometry + scale + metrics |

In [2]:
from __future__ import annotations
from PIL import Image  # must come before any torchvision import (PIL DLL ordering quirk on Windows)

import json
import sys
from pathlib import Path


# ---------------------------------------------------------------------------
# Project root + sys.path setup
# ---------------------------------------------------------------------------

def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "vectorization" / "phase4").exists():
            return candidate
    return here

PROJECT_ROOT = _find_project_root()
R2G_CHECKPOINT = PROJECT_ROOT / "checkpoints_Raster2Graph" / "checkpoint0299.pth"
SEG_CHECKPOINT = PROJECT_ROOT / "checkpoints_CNN" / "segformer_b0_run3" / "best.pt"

for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "external" / "raster_to_graph")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Project root : {PROJECT_ROOT}")
print(f"R2G ckpt     : {R2G_CHECKPOINT} — {'found' if R2G_CHECKPOINT.exists() else 'NOT FOUND'}")
print(f"Seg ckpt     : {SEG_CHECKPOINT} — {'found' if SEG_CHECKPOINT.exists() else 'NOT FOUND'}")


# ---------------------------------------------------------------------------
# Phase 4 pipeline import
# task34: all geometry logic lives in src/vectorization/phase4/pipeline.py
#   Part A — adjusted intervals propagated to SVG/JSON primitives
#   Part B — evidence-based door hinge/swing from red/orange/purple masks
#   Part C — topology-safe endpoint snap before wall chain buffering
# autoreload (cell 1) picks up source edits without a kernel restart.
# ---------------------------------------------------------------------------

from src.vectorization.phase4.pipeline import run_phase4_pipeline


# ---------------------------------------------------------------------------
# Main entry point — thin wrapper around run_phase4_pipeline()
# ---------------------------------------------------------------------------

def vectorize_image(image_path: str, explicit_px_to_mm: float | None = None) -> dict:
    """Run the Phase 4 graph-to-vector pipeline on one floor plan image.

    All pipeline logic is in run_phase4_pipeline(); this function only
    resolves paths, reuses existing graph_pred.json, and formats the summary.
    The segmentation model is cached inside segmentation_inference._SEG_MODEL_CACHE
    so repeated calls on the same kernel do not reload weights from disk.

    Outputs written to the same folder as the input image:
        input.png, image_segmentation.png, image_debug_overlay.png,
        graph_pred.json, graph_pred.svg, graph_overlay.png,
        graph_overlay_aligned.png, final_vector.svg, final_vector.json
    """
    image_path = Path(image_path).expanduser().resolve()
    if not image_path.exists():
        raise FileNotFoundError(f"Input image not found: {image_path}")

    out_dir = image_path.parent
    existing_graph = out_dir / "graph_pred.json"
    run_r2g = not existing_graph.exists()
    if not run_r2g:
        print(f"Reusing existing {existing_graph.name} — skip R2G inference")

    result = run_phase4_pipeline(
        image_path=image_path,
        output_dir=out_dir,
        seg_checkpoint=SEG_CHECKPOINT,
        r2g_checkpoint=R2G_CHECKPOINT,
        explicit_px_to_mm=explicit_px_to_mm,
        run_r2g=run_r2g,
        existing_graph_json=str(existing_graph) if not run_r2g else None,
    )

    wg = result.wall_geometry
    si = result.scale_info
    summary = {
        "input":                    str(image_path),
        "output_dir":               str(out_dir),
        "scale_status":             si.scale_status if si else "unknown",
        "px_to_mm":                 si.px_to_mm if si else None,
        "wall_thickness_mm":        wg.wall_thickness_mm if wg else None,
        "hosted_doors":             len(result.hosted_doors),
        "hosted_windows":           len(result.hosted_windows),
        "rejected_total":           len(result.rejected_openings),
        "wall_chains":              wg.chain_count if wg else 0,
        "disconnected_endpoints":   wg.disconnected_endpoint_count if wg else 0,
        "pre_buffer_nodes":         wg.pre_buffer_node_count if wg else 0,
        "post_snap_nodes":          wg.post_snap_node_count if wg else 0,
        "aligned_edges":            len(result.aligned_graph.get("aligned_edges", [])),
        "elapsed_s":                round(result.elapsed_s, 2),
    }
    print(json.dumps(summary, indent=2))
    return summary

Project root : C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan
R2G ckpt     : C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\checkpoints_Raster2Graph\checkpoint0299.pth — found
Seg ckpt     : C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\checkpoints_CNN\segformer_b0_run3\best.pt — found


In [4]:
# <<< EDIT THIS, then Run All Cells >>>
INPUT_IMAGE_PATH = r"C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\outputs\vectorization\phase4_vectorization\1316\image.png"

vectorize_image(INPUT_IMAGE_PATH)

Reusing existing graph_pred.json — skip R2G inference
[phase4] 1/13 preprocessing...
[phase4] 2/13 loading existing graph_pred.json...
[phase4] 3/13 graph alignment...
[phase4] 4/13 segmentation inference...
[phase4] 5/13 component extraction...
[phase4] 6/13 scale inference...
  scale: estimated px_to_mm=21.21212121212121
[phase4] 7/13 opening detection...
  doors detected: 4 accepted, 0 rejected
  windows detected: 3 accepted, 0 rejected
[phase4] 8/13 opening hosting...
  hosted: 3 doors, 2 windows
  rejected total: 2
[phase4] 9/13 wall interval trimming...
  wall edges after trimming: 16
[phase4] 10/13 wall buffering...
  wall chains: 11, thickness_mm=200.0, disconnected_endpoints=10
  door direction: 3/3 from evidence, 0 fallback
[phase4] 11/13 exporting final SVG...
[phase4] 12/13 exporting final JSON...
[phase4] 13/13 writing debug overlay...

[phase4] done in 0.1s -> C:\Users\kdgki\Desktop\MSCDP\Projects\neural_floorplan\outputs\vectorization\phase4_vectorization\1316
{
  "input

{'input': 'C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_vectorization\\1316\\image.png',
 'output_dir': 'C:\\Users\\kdgki\\Desktop\\MSCDP\\Projects\\neural_floorplan\\outputs\\vectorization\\phase4_vectorization\\1316',
 'scale_status': 'estimated',
 'px_to_mm': 21.21212121212121,
 'wall_thickness_mm': 200.0,
 'hosted_doors': 3,
 'hosted_windows': 2,
 'rejected_total': 2,
 'wall_chains': 11,
 'disconnected_endpoints': 10,
 'pre_buffer_nodes': 22,
 'post_snap_nodes': 19,
 'aligned_edges': 11,
 'elapsed_s': 0.11}